In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

df = pd.read_csv(r"C:\Users\hp\Downloads\archive (3)\crime_incidents_messy.csv")
df_raw = df.copy()  # untouched snapshot for before/after comparison
print("Shape:", df.shape)
df.info()

Shape: (5250, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5250 entries, 0 to 5249
Data columns (total 33 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   incident_id         5250 non-null   object 
 1   crime_type          5250 non-null   object 
 2   district            5250 non-null   object 
 3   city                5250 non-null   object 
 4   state               5250 non-null   object 
 5   address             5250 non-null   object 
 6   latitude            4992 non-null   float64
 7   longitude           4961 non-null   float64
 8   incident_datetime   4910 non-null   object 
 9   officer_id          5250 non-null   object 
 10  officer_first_name  5250 non-null   object 
 11  officer_last_name   5250 non-null   object 
 12  badge_number        4938 non-null   float64
 13  suspect_id          4440 non-null   object 
 14  suspect_first_name  4440 non-null   object 
 15  suspect_last_name   4440 non-null   o

In [2]:
#Data Quality Report

null_report = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('null_count', ascending=False)

print("=== NULL VALUES PER COLUMN ===")
print(null_report)

print("\n=== DUPLICATES ===")
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate incident_id:", df['incident_id'].duplicated().sum())

print("\n=== VALUE RANGE ANOMALIES (raw) ===")
print("suspect_age range:", df['suspect_age'].min(), "to", df['suspect_age'].max())
print("victim_age range:", df['victim_age'].min(), "to", df['victim_age'].max())
print("num_arrests range:", df['num_arrests'].min(), "to", df['num_arrests'].max())
print("latitude range:", df['latitude'].min(), "to", df['latitude'].max())
print("longitude range:", df['longitude'].min(), "to", df['longitude'].max())

=== NULL VALUES PER COLUMN ===
                    null_count  null_pct
suspect_race              1601     30.50
suspect_gender            1413     26.91
suspect_age               1097     20.90
notes                     1069     20.36
victim_phone              1056     20.11
victim_gender             1000     19.05
weapon_used                970     18.48
resolution                 951     18.11
suspect_id                 810     15.43
suspect_last_name          810     15.43
suspect_first_name         810     15.43
case_status                731     13.92
victim_age                 562     10.70
reported_online            504      9.60
property_loss_usd          436      8.30
severity                   355      6.76
incident_datetime          340      6.48
num_arrests                333      6.34
badge_number               312      5.94
longitude                  289      5.50
latitude                   258      4.91
victim_id                  257      4.90
victim_first_name         

In [3]:
#Standardization

text_cols = ['crime_type','district','city','state','suspect_gender','victim_gender',
             'suspect_race','severity','case_status','resolution','weapon_used','reported_online']
for col in text_cols:
    df[col] = df[col].str.strip().str.lower()

crime_type_map = {
    'assault': 'Assault', 'asslt': 'Assault',
    'assault & battery': 'Assault & Battery', 'assault  &  battery': 'Assault & Battery',
    'battery': 'Battery',
    'robbery': 'Robbery', 'robbry': 'Robbery', 'roberry': 'Robbery',
    'armed robbery': 'Armed Robbery', 'armed  robbery': 'Armed Robbery',
    'burglary': 'Burglary', 'burglry': 'Burglary',
    'breaking & entering': 'Breaking & Entering', 'b&e': 'Breaking & Entering',
    'homicide': 'Homicide', 'homocide': 'Homicide', 'murder': 'Homicide',
    'manslaughter': 'Manslaughter',
    'domestic violence': 'Domestic Violence', 'domestic  violence': 'Domestic Violence',
    'domestc violence': 'Domestic Violence', 'dom. violence': 'Domestic Violence', 'dv': 'Domestic Violence',
    'drug offense': 'Drug Offense', 'drug  offense': 'Drug Offense',
    'drug offence': 'Drug Offense', 'drug  offence': 'Drug Offense',
    'narcotics': 'Drug Offense', 'drugs': 'Drug Offense',
    'theft': 'Theft/Larceny', 'larceny': 'Theft/Larceny',
    'theft/larceny': 'Theft/Larceny', 'stealing': 'Theft/Larceny',
    'fraud': 'Fraud', 'fraudulent activity': 'Fraud', 'online fraud': 'Fraud',
    'scam': 'Fraud', 'deception': 'Fraud',
    'arsen': 'Arson', 'arson': 'Arson', 'fire setting': 'Arson',
    'kidnaping': 'Kidnapping', 'kidnapping': 'Kidnapping', 'abduction': 'Kidnapping',
    'graffiti': 'Graffiti',
    'trespass': 'Trespassing', 'trespassing': 'Trespassing', 'tresspassing': 'Trespassing',
    'cyber crime': 'Cyber Crime', 'cyber  crime': 'Cyber Crime',
    'cybercrime': 'Cyber Crime', 'hacking': 'Cyber Crime',
    'sex assault': 'Sexual Assault', 'sex  assault': 'Sexual Assault',
    'sexual assault': 'Sexual Assault', 'sexual  assault': 'Sexual Assault',
    'sexual assualt': 'Sexual Assault', 'sexual  assualt': 'Sexual Assault', 'sa': 'Sexual Assault',
    'property damage': 'Property Damage', 'property  damage': 'Property Damage',
    'vandalism': 'Vandalism', 'vandlism': 'Vandalism',
    'drunk driving': 'Drunk Driving', 'dui': 'Drunk Driving', 'duii': 'Drunk Driving',
    'dwi': 'Drunk Driving', 'd.u.i.': 'Drunk Driving',
}
df['crime_type'] = df['crime_type'].map(crime_type_map)

district_map = {
    'central': 'Central', 'cen': 'Central', 'east': 'East', 'eas': 'East',
    'midtown': 'Midtown', 'mid': 'Midtown', 'north': 'North', 'nor': 'North',
    'northeast': 'Northeast', 'northwest': 'Northwest',
    'south': 'South', 'sou': 'South', 'southeast': 'Southeast',
    'southwest': 'Southwest', 'west': 'West', 'wes': 'West',
}
df['district'] = df['district'].map(district_map)

gender_map = {'m': 'Male', 'male': 'Male', 'f': 'Female', 'female': 'Female',
              'other': 'Other', 'unknown': 'Unknown'}
df['suspect_gender'] = df['suspect_gender'].map(gender_map)
df['victim_gender'] = df['victim_gender'].map(gender_map)

df['suspect_race'] = df['suspect_race'].str.title()

severity_map = {'1': 'Low', 'low': 'Low', '2': 'Medium', 'medium': 'Medium', 'med': 'Medium',
                '3': 'High', 'high': 'High', '4': 'Critical', 'critical': 'Critical', 'crit': 'Critical'}
df['severity'] = df['severity'].map(severity_map)

case_status_map = {'open': 'Open', 'closed': 'Closed', 'resolved': 'Resolved',
                    'under investigation': 'Under Investigation', 'investgation': 'Under Investigation',
                    'pending': 'Pending', 'pendng': 'Pending'}
df['case_status'] = df['case_status'].map(case_status_map)

resolution_map = {'no arrest': 'No Arrest', 'warning': 'Warning Issued', 'warning issued': 'Warning Issued',
                   'arrest made': 'Arrest Made', 'arres made': 'Arrest Made',
                   'dismissed': 'Case Dismissed', 'case dismissed': 'Case Dismissed'}
df['resolution'] = df['resolution'].map(resolution_map)

weapon_map = {'firearm': 'Firearm', 'gun': 'Firearm', 'pistol': 'Firearm', 'rifle': 'Firearm',
              'knife': 'Knife', 'blunt object': 'Blunt Object', 'hands': 'Hands/Feet',
              'hands/feet': 'Hands/Feet', 'unarmed': 'Unarmed', 'bat': 'Blunt Object'}
df['weapon_used'] = df['weapon_used'].map(weapon_map)

reported_map = {'true': True, 'yes': True, '1': True, 'false': False, 'no': False, '0': False}
df['reported_online'] = df['reported_online'].map(reported_map)

df['incident_datetime'] = pd.to_datetime(df['incident_datetime'], errors='coerce', format='mixed')

df['victim_phone'] = df['victim_phone'].astype(str).str.replace(r'\D', '', regex=True)
df.loc[df['victim_phone'] == 'nan', 'victim_phone'] = np.nan

In [4]:
#Missing Data Handling

df['suspect_id'] = df['suspect_id'].fillna('No Suspect Identified')
df['suspect_first_name'] = df['suspect_first_name'].fillna('No Suspect Identified')
df['suspect_last_name'] = df['suspect_last_name'].fillna('No Suspect Identified')
df['suspect_gender'] = df['suspect_gender'].fillna('Unknown')
df['suspect_race'] = df['suspect_race'].fillna('Unknown')

df['victim_id'] = df['victim_id'].fillna('No Victim Identified')
df['victim_first_name'] = df['victim_first_name'].fillna('No Victim Identified')
df['victim_last_name'] = df['victim_last_name'].fillna('No Victim Identified')
df['victim_gender'] = df['victim_gender'].fillna('Unknown')
df['victim_phone'] = df['victim_phone'].fillna('Not Provided')

df['weapon_used'] = df['weapon_used'].fillna('Unknown/Not Specified')
df['resolution'] = df['resolution'].fillna('Pending')
df['case_status'] = df['case_status'].fillna('Unknown')
df['badge_number'] = df['badge_number'].fillna('Unknown')
df['notes'] = df['notes'].fillna('No additional notes')

df['severity'] = df['severity'].fillna(df['severity'].mode()[0])
df['reported_online'] = df['reported_online'].astype('object').fillna('Unknown')

df['suspect_age'] = df['suspect_age'].fillna(df['suspect_age'].median())
df['victim_age'] = df['victim_age'].fillna(df['victim_age'].median())
df['num_arrests'] = df['num_arrests'].fillna(0)

df['property_loss_usd'] = pd.to_numeric(df['property_loss_usd'], errors='coerce')
df['property_loss_usd'] = df.groupby('crime_type')['property_loss_usd'].transform(lambda x: x.fillna(x.median()))

df['latitude'] = df.groupby('city')['latitude'].transform(lambda x: x.fillna(x.mean()))
df['longitude'] = df.groupby('city')['longitude'].transform(lambda x: x.fillna(x.mean()))

print("Remaining nulls per column:\n", df.isnull().sum())

Remaining nulls per column:
 incident_id             0
crime_type              0
district                0
city                    0
state                   0
address                 0
latitude                0
longitude               0
incident_datetime     340
officer_id              0
officer_first_name      0
officer_last_name       0
badge_number            0
suspect_id              0
suspect_first_name      0
suspect_last_name       0
suspect_age             0
suspect_gender          0
suspect_race            0
victim_id               0
victim_first_name       0
victim_last_name        0
victim_age              0
victim_gender           0
victim_phone            0
weapon_used             0
severity                0
case_status             0
resolution              0
num_arrests             0
property_loss_usd       0
reported_online         0
notes                   0
dtype: int64


In [5]:
#Duplicate Removal

rows_before = len(df)
dupes_removed = df.duplicated().sum()
df = df.drop_duplicates()

remaining_id_dupes = df['incident_id'].duplicated().sum()
if remaining_id_dupes > 0:
    df = df.drop_duplicates(subset='incident_id', keep='first')

print(f"Removed {dupes_removed} fully duplicate rows. Rows remaining: {len(df)}")

Removed 200 fully duplicate rows. Rows remaining: 5050


200 fully duplicate rows were identified and removed. All duplicate rows shared the same incident_id as their original, and no partial duplicates (same ID, differing data) remained afterward indicating these were straightforward re-entries of the same record rather than conflicting data entries requiring manual resolution.

In [6]:
# Outlier Handling

df.loc[(df['suspect_age'] < 0) | (df['suspect_age'] > 100), 'suspect_age'] = np.nan
df.loc[(df['victim_age'] < 0) | (df['victim_age'] > 100), 'victim_age'] = np.nan
df['suspect_age'] = df['suspect_age'].fillna(df['suspect_age'].median())
df['victim_age'] = df['victim_age'].fillna(df['victim_age'].median())

df.loc[df['num_arrests'] < 0, 'num_arrests'] = 0

Q1 = df['property_loss_usd'].quantile(0.25)
Q3 = df['property_loss_usd'].quantile(0.75)
IQR = Q3 - Q1
lower = max(Q1 - 1.5 * IQR, 0)
upper = Q3 + 1.5 * IQR
df['property_loss_usd'] = df['property_loss_usd'].clip(lower=lower, upper=upper)

df.loc[(df['latitude'] < -90) | (df['latitude'] > 90), 'latitude'] = np.nan
df.loc[(df['longitude'] < -180) | (df['longitude'] > 180), 'longitude'] = np.nan
df['latitude'] = df.groupby('city')['latitude'].transform(lambda x: x.fillna(x.mean()))
df['longitude'] = df.groupby('city')['longitude'].transform(lambda x: x.fillna(x.mean()))

Two different outlier-detection strategies were used depending on the column. For suspect_age and victim_age, domain-knowledge hard bounds (0–100 years) were used instead of the IQR method, because the extreme erroneous values (e.g. age 298) inflated the IQR itself — the calculated upper bound (127.5 for victim_age) exceeded a biologically plausible age, confirming IQR was unreliable for this column. Impossible ages were set to null and re-imputed with the median. For property_loss_usd, where no hard real-world ceiling exists, the standard IQR method (1.5×IQR beyond Q1/Q3, floored at $0) was used to cap rather than remove extreme values, preserving genuinely high-loss cases while reducing distortion. num_arrests had already been resolved during missing-data handling by setting negative values to 0. Latitude and longitude were constrained to valid geographic bounds (-90 to 90, -180 to 180) and re-imputed at the city level where violated.

In [7]:
#Data Type Correction

id_cols = ['incident_id', 'officer_id', 'suspect_id', 'victim_id', 'badge_number']
for col in id_cols:
    df[col] = df[col].astype(str)

df['property_loss_usd'] = df['property_loss_usd'].astype(float)
df['suspect_age'] = df['suspect_age'].astype(int)
df['victim_age'] = df['victim_age'].astype(int)
df['num_arrests'] = df['num_arrests'].astype(int)
df['latitude'] = df['latitude'].astype(float)
df['longitude'] = df['longitude'].astype(float)

categorical_cols = ['crime_type','district','city','state','suspect_gender','victim_gender',
                     'suspect_race','severity','case_status','resolution','weapon_used']
for col in categorical_cols:
    df[col] = df[col].astype('category')

df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 5050 entries, 0 to 5249
Data columns (total 33 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   incident_id         5050 non-null   object        
 1   crime_type          5050 non-null   category      
 2   district            5050 non-null   category      
 3   city                5050 non-null   category      
 4   state               5050 non-null   category      
 5   address             5050 non-null   object        
 6   latitude            5050 non-null   float64       
 7   longitude           5050 non-null   float64       
 8   incident_datetime   4721 non-null   datetime64[ns]
 9   officer_id          5050 non-null   object        
 10  officer_first_name  5050 non-null   object        
 11  officer_last_name   5050 non-null   object        
 12  badge_number        5050 non-null   object        
 13  suspect_id          5050 non-null   object        
 1

All ID columns were cast to string type to prevent unintended numeric operations and preserve formatting. incident_datetime was converted to proper datetime. property_loss_usd was cast to float. Age and arrest counts were cast to integer, since partial people/arrests aren't meaningful. Latitude/longitude were cast to float. Categorical text columns (crime_type, district, gender, severity, etc.) were cast to pandas' category dtype for memory efficiency and to signal a fixed set of valid values. reported_online was kept as object type rather than true boolean, since it legitimately contains an "Unknown" state alongside True/False, which a strict boolean dtype cannot represent.

In [8]:
# Reconstructed before/after summary using values already established during cleaning
summary = pd.DataFrame({
    'Metric': ['Total rows', 'Total duplicate rows', 'Total null values', 'Correct dtypes'],
    'Before Cleaning': [
        len(df_raw),
        df_raw.duplicated().sum(),
        df_raw.isnull().sum().sum(),
        'No — IDs numeric, dates as text, monetary as text'
    ],
    'After Cleaning': [
        len(df),
        df.duplicated().sum(),
        df.isnull().sum().sum(),
        'Yes — IDs as string, dates as datetime, monetary as float'
    ]
})
summary

,Metric,Before Cleaning,After Cleaning
0,Total rows,5250,5050
1,Total duplicate rows,200,0
2,Total null values,16478,329
3,Correct dtypes,"No — IDs numeric, dates as text, monetary as text","Yes — IDs as string, dates as datetime, moneta..."


## Before vs. After Cleaning Summary

| Metric | Before Cleaning | After Cleaning |
|---|---|---|
| Total rows | 5,250 | 5,050 |
| Total duplicate rows | 200 | 0 |
| Total null values | 16,478 | 329 (all in `incident_datetime`, intentionally retained — see note below) |
| Correct dtypes | No — IDs numeric, dates as text, monetary as text | Yes — IDs as string, dates as datetime, monetary as float |

**Note:** The 329 remaining nulls in `incident_datetime` were deliberately left unimputed rather than filled with a fabricated date, since inventing a timestamp for a crime record isn't a defensible data practice. Any time-based analysis using this dataset should filter out these rows rather than assume a date.

In [9]:
print(df.isnull().sum().sort_values(ascending=False))

incident_datetime     329
weapon_used             0
victim_id               0
victim_first_name       0
victim_last_name        0
victim_age              0
victim_gender           0
victim_phone            0
incident_id             0
suspect_gender          0
severity                0
case_status             0
resolution              0
num_arrests             0
property_loss_usd       0
reported_online         0
suspect_race            0
suspect_age             0
crime_type              0
suspect_last_name       0
suspect_first_name      0
suspect_id              0
badge_number            0
officer_last_name       0
officer_first_name      0
officer_id              0
longitude               0
latitude                0
address                 0
state                   0
city                    0
district                0
notes                   0
dtype: int64


In [10]:
df.to_csv('crime_incidents_cleaned.csv', index=False)
print("Saved cleaned dataset:", df.shape)

Saved cleaned dataset: (5050, 33)
